In [ ]:
import numpy as np

In [ ]:
node_hamming_weights = [1,1,1,1,1]
hamming_weights = 7
size = 5


def add_constraint(node_hamming_weights, hamming_weights):
    size = len(node_hamming_weights)
    Q = np.zeros((size, size))
    for i in range(size):
        for j in range(size):
            if i == j:
                Q[i, j] = node_hamming_weights[i]**2 - 2*hamming_weights*node_hamming_weights[i]
            else:
                Q[i, j] = node_hamming_weights[i]*node_hamming_weights[j]
    
    return Q

In [ ]:
import copy
full_list = []

same_list = [(1,2),(3,4)]
diff_list = [(2,3)]


list_seq = [[i+1] for i in range(12)]
list_seq += [[1,2],[2,3],[3,4],[5,7]]


def make_check(list_seq):
    list_seq = copy.deepcopy(list_seq)
    full_list = []
    add_comp = None
    for seq in list_seq:
        if len(full_list) == 0:
            full_list =[seq]
        else:
            break_comp = False
            seq_test = abs(np.array(seq))
            for list_comp in full_list:
                if break_comp:
                    break
                for index,comp in enumerate(list_comp):
                    if break_comp:
                        break
                    for test_idx,test in enumerate(seq_test):
                        if test == abs(comp):
                            test = seq.pop(test_idx)
                            add_comp = np.sign(test)*np.sign(comp)*np.array(seq)
                            break_comp = True
                            list_comp+=list(add_comp)
                            break
                                
                
                
                
            if add_comp is None and len(seq) >= 1:
                full_list.append(seq)
            add_comp = None
    return full_list

full_list = make_check(list_seq)
while(full_list != make_check(full_list)):
    full_list = make_check(full_list)
print(f'full_list : {full_list}')


In [ ]:
def reduce_hamiltonian(Q_full, check_list):
    """
    Reduces the given Hamiltonian matrix by applying the constraint Z_k = sign * Z_l.

    Args:
        J (np.array): The initial Hamiltonian matrix (including diagonal terms).
        k (int): Index of the variable to be removed.
        l (int): Index of the variable to be replaced.
        node_assignments (dict): Dictionary mapping node indices to assignments.
        sign (int): Relationship (1 if identical, -1 if opposite).

    Returns:
        tuple: 
            - np.array: The reduced Hamiltonian matrix with the k-th variable removed.
            - np.array: An expanded version of the reduced matrix with extra rows and columns added back.
    """
    # Update interactions: J[i, l] = J[i, l] + sign * J[i, k]
 
    Q_full_copy = copy.deepcopy(Q_full)
    Q_full_copy = (Q_full_copy+Q_full_copy.T)/2
    for check in check_list:
        check_value = np.sign(check)*(abs(np.array(check))-1)

        if len(check)==1:
            pass
        else:
            for value in check_value[1:]:

                Q_full_copy[int(abs(check_value[0])),int(abs(check_value[0]))] += np.sign(value)*Q_full_copy[int(abs(value)),int(abs(value))] 
                for i in range(len(Q_full_copy)):
                    if i != int(abs(check_value[0])) and i != int(abs(value)):
                        Q_full_copy[int(abs(check_value[0])),i] += np.sign(value)*Q_full_copy[int(abs(value)),i]
                        Q_full_copy[i,int(abs(check_value[0]))] += np.sign(value)*Q_full_copy[i,int(abs(value))]
    keep_indices = []
    for check in check_list:
        check_value = np.sign(check)*(abs(np.array(check))-1)
        keep_indices.append(abs(int(check_value[0])))
    Q_full_copy = np.array(Q_full_copy)[np.ix_(keep_indices, keep_indices)]
    R = copy.deepcopy(Q_full_copy)

    remove_list = [x for x in range(len(Q_full)) if x not in keep_indices]
    # Add back zero rows and columns at specified indices
    for key in remove_list:
        R = add_zero_row_col(R, key)

                    

    return Q_full_copy, R

def add_zero_row_col(matrix, m):
    """
    Adds a new row and column filled with zeros at the specified position 
    in an n x n matrix, resulting in an (n+1) x (n+1) matrix.

    Args:
        matrix (np.array): The original n x n matrix.
        m (int): The index (0-based) where the new row and column will be inserted.

    Returns:
        np.array: The expanded (n+1) x (n+1) matrix with the new row and column filled with zeros.
    """
    n = matrix.shape[0]

    # Create a new (n+1)x(n+1) matrix initialized with zeros
    new_matrix = np.zeros((n + 1, n + 1))

    # Copy the existing elements to the new matrix
    new_matrix[:m, :m] = matrix[:m, :m]      # Top-left block
    new_matrix[:m, m+1:] = matrix[:m, m:]    # Top-right block
    new_matrix[m+1:, :m] = matrix[m:, :m]    # Bottom-left block
    new_matrix[m+1:, m+1:] = matrix[m:, m:]  # Bottom-right block

    return new_matrix

In [ ]:
def make_node_weights(full_list):
    node_weights = []
    hamming_weights_default = 0
    for seq in full_list:
        node_weights.append(np.sum(np.array(seq) >= 0) - np.sum(np.array(seq) < 0))
        hamming_weights_default += np.sum(np.array(seq) < 0)
    return node_weights, hamming_weights_default

In [ ]:
node_weights, hamming_weights_default = make_node_weights(full_list)

In [ ]:
hamming_weights

In [ ]:
Q = add_constraint(node_weights, hamming_weights-hamming_weights_default)

In [ ]:
from itertools import product

inputs = product(*[[-1,1] for i in range(len(Q))])

In [ ]:
qubo_to_ising(Q)

In [ ]:
res_dict = {}
for value in inputs:
    res = value@Q@value
    res = state_energy(np.array(value),qubo_to_ising(Q))
    res_dict[value] = res

In [ ]:
res_dict = sorted(res_dict.items(),key = lambda item: item[1])

In [ ]:
res_dict

In [ ]:
random.shuffle(res_dict)

In [ ]:
outputs = {}
outputs_2 = []
for index,key in enumerate(res_dict):
    val = np.array(key[0])
    val = (1-np.array(key[0]))/2
    outputs[np.array(node_weights)@val] = key[1]
    outputs_2.append([np.sum(key[0]),key[1]])
print(hamming_weights-hamming_weights_default)

In [ ]:
outputs

In [ ]:
from itertools import combinations

def generate_all_combinations(lst, n):
    """
    리스트에서 n개의 1을 -1로 바꾼 모든 조합을 생성한다.
    
    :param lst: 1로만 구성된 리스트
    :param n: -1로 바꿀 개수 (항상 5로 가정)
    :return: 모든 가능한 조합 리스트
    """
    indices = [i for i, x in enumerate(lst) if x == 1]  # 1의 인덱스 찾기
    if len(indices) < n:
        return []  # 1의 개수가 n보다 적으면 조합을 만들 수 없음
    
    all_combinations = []
    
    for comb in combinations(indices, n):  # 5개를 선택하는 모든 조합 생성
        new_lst = lst[:]  # 원본 리스트 복사
        for i in comb:
            new_lst[i] = -1  # 선택된 인덱스를 -1로 변경
        all_combinations.append(new_lst)
    
    return all_combinations

# 사용 예시
size = 12
lst = [1] * size  # 길이 10짜리 1로만 구성된 리스트






In [ ]:
from rl_qaoa import generate_upper_triangular_qubo
Q = generate_upper_triangular_qubo(size, (-1,-3), (2,4), integer=False)

In [ ]:
import itertools
def state_energy(state, Q):
        """
        Computes the energy of a given state based on the QUBO matrix.

        Parameters
        ----------
        state : np.ndarray
            Binary state vector (e.g. [-1, 1, -1, 1]).

        Q : np.ndarray
            The QUBO matrix representing the optimization problem.

        Returns
        -------
        float
            The computed energy value of the given state.
        """
        # Create an identity matrix of the same size
        identity_matrix = np.eye(Q.shape[0], dtype=bool)

        # Remove diagonal elements from the QUBO matrix to isolate interactions
        interaction = np.where(identity_matrix, 0, Q)
        diagonal_elements = np.diag(Q)

        # Compute the energy using the QUBO formulation
        value = diagonal_elements @ state + state.T @ interaction @ state
        return value


full_list = [[i+1] for i in range(size)]
configs = list(itertools.product([-1, 1], repeat=len(full_list)))

cases = []
for config in configs:
    res = []
    for index,value in enumerate(full_list):
        res += list(config[index]*np.sign(value))

    cases.append(np.array(res))


best_value_full = np.inf
res_node_full = None
full_value_seq = {}
# Find all valid combinations considering the same and different constraints
for comb in cases:
    value = state_energy(np.array(comb), Q)
    if value < best_value_full:
        best_value_full = value
        res_node_full = copy.copy(comb)
    full_value_seq[f"{list(comb)}"] = value


In [ ]:
def qubo_to_ising(Q):
    """
    0/1 basis의 QUBO matrix Q (대칭 np.array)를 받아,
    -1/1 basis (Ising)로 변환하여 linear term h와 quadratic interaction matrix J를 반환합니다.
    
    x_i = (1 - z_i) / 2 변환을 이용.
    
    변환된 Ising Hamiltonian:
       H(z) = constant + sum_i h_i * z_i + sum_{i<j} J_{ij} * z_i * z_j.
    
    Parameters:
        Q (np.array): QUBO matrix (대칭, diagonal에 1-body term, off-diagonal에 2-body interaction)

    Returns:
        h (np.array): Ising 모델의 선형 항 계수 (길이 n)
        J (np.array): Ising 모델의 상호작용 계수 (n x n, 대칭, diagonal은 0)
    """
    n = Q.shape[0]
    h = np.zeros(n)
    J = np.zeros((n, n))
    # i<j에 대해 quadratic interaction term 계산
    for i in range(n):
        for j in range(n):
          if i!=j:
            h[i] -= Q[i, j] / 4.0
            h[j] -= Q[i, j] /4
            J[i, j] = Q[i, j] / 4.0

    for i in range(n):
        h[i] -= Q[i, i] / 2.0
    Q_res = J
    np.fill_diagonal(Q_res, h)
    return Q_res

In [ ]:
res_node_full

In [ ]:
import copy
full_list = []

same_list = [(1,2),(3,4)]
diff_list = [(2,3)]


list_seq = [[i+1] for i in range(size)]
list_seq += [[2,3],[11,12]]


def make_check(list_seq):
    list_seq = copy.deepcopy(list_seq)
    full_list = []
    add_comp = None
    for seq in list_seq:
        if len(full_list) == 0:
            full_list =[seq]
        else:
            break_comp = False
            seq_test = abs(np.array(seq))
            for list_comp in full_list:
                if break_comp:
                    break
                for index,comp in enumerate(list_comp):
                    if break_comp:
                        break
                    for test_idx,test in enumerate(seq_test):
                        if test == abs(comp):
                            test = seq.pop(test_idx)
                            add_comp = np.sign(test)*np.sign(comp)*np.array(seq)
                            break_comp = True
                            list_comp+=list(add_comp)
                            break
                                
                
                
                
            if add_comp is None and len(seq) >= 1:
                full_list.append(seq)
            add_comp = None
    return full_list

full_list = make_check(list_seq)
while(full_list != make_check(full_list)):
    full_list = make_check(full_list)
print(f'full_list : {full_list}')


In [ ]:
reduce_ham,_ = reduce_hamiltonian(Q,full_list)

In [ ]:
import itertools
def state_energy(state, Q):
        """
        Computes the energy of a given state based on the QUBO matrix.

        Parameters
        ----------
        state : np.ndarray
            Binary state vector (e.g. [-1, 1, -1, 1]).

        Q : np.ndarray
            The QUBO matrix representing the optimization problem.

        Returns
        -------
        float
            The computed energy value of the given state.
        """
        # Create an identity matrix of the same size
        identity_matrix = np.eye(Q.shape[0], dtype=bool)

        # Remove diagonal elements from the QUBO matrix to isolate interactions
        interaction = np.where(identity_matrix, 0, Q)
        diagonal_elements = np.diag(Q)
        # Compute the energy using the QUBO formulation
        value = diagonal_elements @ state + state.T @ interaction @ state
        return value



configs = list(itertools.product([-1, 1], repeat=len(full_list)))

cases = []
for config in configs:
    res = []
    for index,value in enumerate(full_list):
        res += list(config[index]*np.sign([value[0]]))

    cases.append(np.array(res))


best_value = np.inf
res_node = None
value_seq = {}
# Find all valid combinations considering the same and different constraints
for comb in cases:
    value = state_energy(np.array(comb), reduce_ham)
    if value < best_value:
        best_value = value
        res_node = copy.copy(comb)
    value_seq[f'{list(comb)}'] = value


In [ ]:
value_seq

In [ ]:
full_list

In [ ]:
best_value,res_node

In [ ]:
best_value_full,res_node_full

In [ ]:
res_node_full

In [ ]:
value_seq

In [ ]:
from rl_qaoa import zero_lower_triangle
import json
import numpy as np
with open("./data/matrices7by7.json", "r") as f:
    matrices_data = json.load(f)

In [ ]:
def data_to_QUBO(matrix,hamming_weight):
    return -np.diag([1]*len(matrix))+matrix/hamming_weight

In [ ]:
data_to_QUBO(np.array(matrices_data[0]),3)

In [8]:
import pennylane as qml
from pennylane import numpy as np
from scipy.optimize import minimize
from modules.rl_qaoa import *
import random
import json
# Define QAOA depth
def data_to_QUBO(matrix,hamming_weight,l):
    return -np.diag([1]*len(matrix))+matrix/hamming_weight*l
def QpasqalOptimized(matrix, number):
    res = copy.deepcopy(matrix)
    N = res.shape[0]
    for i in range(N):
        res[i][i] = 0
    # 대각 성분 0으로 만들기
    for i in range(N):
        for j in range(i + 1, N):
            res[i][j] = res[i][j] + (matrix[i][i] + matrix[j][j]) / ((number - 1) * 2)
            res[j][i] = res[j][i] + (matrix[i][i] + matrix[j][j]) / ((number - 1) * 2)


    return res

depth =1
size = 7
seed = 50
hamming_weight = int(size/2)
penalty = 2
with open(f"./data/matrices{size}by{size}.json", "r") as f:
    matrices_data = json.load(f)
""" with open(f"./data/GeoWindSunMatrices7by7.json", "r") as f:
    matrices_data = json.load(f) """
# Generate a QUBO matrix that is challenging for classical QAOA optimization
np.random.seed(seed)
Q = data_to_QUBO(np.array(matrices_data[4]),hamming_weight,3)
Q_cal = zero_lower_triangle(Q + add_constraint([1]*size,hamming_weight)*penalty)


In [ ]:
## Optimization with randomstate
# Initial parameters for QAOA
n = Q.shape[0]
n_c = 2

np.random.seed(seed)
init_params = np.reshape(np.array([0.20638753, 0.15790969]*(n-n_c)),-1)


# RL-QAOA setup
rl_qaoa = RL_QAOA(Q_cal/2+Q_cal.T/2,n,init_params,b_vector = np.array([[25.*((n-i)/10+1)]*int((n**2)) for i in range(n-n_c)]),QAOA_depth=depth,gamma = 0.99,learning_rate_init=[0.05,0.0])
final_config = rl_qaoa.rqaoa_execute()
rl_qaoa.n_c =n_c 
print(f"classical_result : {float(final_config[2])},best : {rl_qaoa.node_assignments}" )


# Execute RQAOA
final_config = rl_qaoa.RL_QAOA(episodes=50,epochs=100,log_interval=1,correct_ans=float(final_config[2]))

classical_result : -3.850997508931143,best : [1, -1, 1, 1, 1, -1, -1]


Epoch 1/100:   0%|          | 0/50 [00:00<?, ? episode/s]

Epoch 1/100: 100%|██████████| 50/50 [00:17<00:00,  2.92 episode/s]


  Probability of finding correct solution: 0.1200
  Average reward: -3.7299067390099885
  Lowest reward obtained: -3.850997508931143
  Best state at lowest value: [ 1 -1  1  1  1 -1 -1]
  number of nodes : 187


Epoch 2/100: 100%|██████████| 50/50 [00:13<00:00,  3.78 episode/s]


...

  Probability of finding correct solution: 0.0200
  Average reward: -3.719510568367241
  Lowest reward obtained: -3.8412967607103905
  Best state at lowest value: [ 1 -1  1 -1  1  1 -1]
  number of nodes : 5131


Epoch 29/100: 100%|██████████| 50/50 [00:15<00:00,  3.20 episode/s]


  Probability of finding correct solution: 0.0200
  Average reward: -3.7132319868620796
  Lowest reward obtained: -3.8412967607103905
  Best state at lowest value: [ 1 -1  1 -1  1  1 -1]
  number of nodes : 5309


Epoch 30/100:  26%|██▌       | 13/50 [00:06<00:18,  1.97 episode/s]


KeyboardInterrupt: 

In [13]:
## Optimization with randomstate
# Initial parameters for QAOA
n = Q.shape[0]
n_c = 2

np.random.seed(seed)
init_params = np.reshape(np.array([0.20638753, 0.15790969]*(n-n_c)),-1)


# RL-QAOA setup
rl_qaoa = RL_QAOA_constraint(Q,n,init_params,b_vector = np.array([[25.*((n-i)/10+1)]*int((n**2)) for i in range(n-n_c)]),hamming_weight= hamming_weight,penalty=2,QAOA_depth=depth,gamma = 0.99,learning_rate_init=[0.001 ,0.0])
final_config = rl_qaoa.rqaoa_execute()
rl_qaoa.n_c =n_c
print(f"classical_result : {float(final_config[2])},best : {rl_qaoa.node_assignments}" )


# Execute RQAOA
final_config = rl_qaoa.RL_QAOA(episodes=50,epochs=100,log_interval=1,correct_ans=float(final_config[2]))

classical_result : -2.852374045814237,best : [1, -1, 1, 1, 1, -1, -1]


Epoch 1/100: 100%|██████████| 50/50 [00:20<00:00,  2.49 episode/s]


  Probability of finding correct solution: 0.0200
  Average reward: -2.697140525283488
  Lowest reward obtained: -2.852374045814237
  Best state at lowest value: [ 1 -1  1  1  1 -1 -1]
  passed episodes : 47
  number of nodes : 197
  99.99% solution : 0.0200


Epoch 2/100: 100%|██████████| 50/50 [00:19<00:00,  2.59 episode/s]


  Probability of finding correct solution: 0.0600
  Average reward: -2.7097600193046216
  Lowest reward obtained: -2.852374045814237
  Best state at lowest value: [ 1 -1  1  1  1 -1 -1]
  passed episodes : 48
  number of nodes : 401
  99.99% solution : 0.0600


...

  Probability of finding correct solution: 0.0600
  Average reward: -2.7503681468301533
  Lowest reward obtained: -2.852374045814237
  Best state at lowest value: [ 1 -1  1  1  1 -1 -1]
  passed episodes : 48
  number of nodes : 2546
  99.99% solution : 0.0600


Epoch 14/100: 100%|██████████| 50/50 [00:18<00:00,  2.76 episode/s]


  Probability of finding correct solution: 0.0600
  Average reward: -2.70132622413139
  Lowest reward obtained: -2.852374045814237
  Best state at lowest value: [ 1 -1  1  1  1 -1 -1]
  passed episodes : 47
  number of nodes : 2744
  99.99% solution : 0.0600


Epoch 15/100:  14%|█▍        | 7/50 [00:04<00:27,  1.54 episode/s]


KeyboardInterrupt: 

In [ ]:
rl_qaoa.b

In [ ]:
## Optimization with randomstate
# Initial parameters for QAOA
n = Q.shape[0]
n_c = 2

np.random.seed(seed)
init_params = np.reshape(np.array([-0.12107375,  0.24919166]*(n-n_c)),-1)


# RL-QAOA setup
rl_qaoa = RL_QAOA_constraint(Q,n,init_params,b_vector = np.array([[100.*((n-i)/10+1)]*int((n**2)) for i in range(n-n_c)]),hamming_weight= hamming_weight,penalty=1.5,QAOA_depth=depth,gamma = 0.99,learning_rate_init=[0.005,0.5])
final_config = rl_qaoa.rqaoa_execute()
rl_qaoa.n_c =n_c
print(f"classical_result : {float(final_config[2])},best : {rl_qaoa.node_assignments}" )


# Execute RQAOA
final_config = rl_qaoa.RL_QAOA(episodes=50,epochs=100,log_interval=5,correct_ans=float(final_config[2]))

In [ ]:
zero_lower_triangle(Q)